# Section 1 — literature retrieval, topic modelling, and ontology-guided extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/romenmeitei/AISKG_01_Framework/blob/main/Mushroom_KG_Upstream_Literature_to_Extraction_Pipeline_v1.ipynb)

This notebook is the **upstream first section** of the reproducibility repository. It precedes the canonical post-extraction workflow.

- **`MANUSCRIPT_SNAPSHOT` (default, tested):** verifies the frozen corpus and expert-curation inputs, replays topic consolidation, regenerates the ontology-guided entity and relation tables, audits them against the frozen study outputs, and creates a complete input ZIP for Section 2.
- **`LIVE_REFRESH` (optional):** runs the recorded PubMed search and, when licensed API credentials are available, Scopus and Web of Science retrieval; performs harmonization and deduplication; fits `all-mpnet-base-v2` + UMAP + HDBSCAN + BERTopic; and creates an expert-review template. A second live phase applies the completed topic review and regenerates semantic extraction.

The frozen snapshot is the authoritative route for the manuscript numbers. A live refresh is expected to differ because bibliographic databases and software models change over time.

In [ ]:
# ================================ CONFIGURATION ================================
RUN_MODE = "MANUSCRIPT_SNAPSHOT"   # "MANUSCRIPT_SNAPSHOT" or "LIVE_REFRESH"
LIVE_PHASE = "RETRIEVE_AND_MODEL"  # or "APPLY_CURATION_AND_EXTRACT"
LIVE_EXPERT_REVIEW_FILE = ""       # path to completed live topic-review CSV
FALLBACK_TO_SNAPSHOT = True         # use exact snapshot if a live service fails
AUTO_DOWNLOAD_OUTPUTS = True

INPUT_ZIP_NAME = "Mushroom_KG_Upstream_Inputs_v1.zip"
OUTPUT_ZIP_NAME = "Mushroom_KG_Upstream_Reproducibility_Outputs.zip"
BRIDGE_ZIP_NAME = "Mushroom_KG_Reproducibility_Inputs_v2_from_upstream.zip"

print("Run mode:", RUN_MODE)
if RUN_MODE.upper() == "LIVE_REFRESH":
    print("Live phase:", LIVE_PHASE)

In [ ]:
# Install only packages that are missing. The default snapshot mode does not
# download language models or contact external databases.
import importlib.util
import subprocess
import sys

snapshot_requirements = {
    "pandas": "pandas>=2.2,<3",
    "numpy": "numpy>=1.26,<3",
    "scipy": "scipy>=1.13,<2",
    "spacy": "spacy>=3.8,<3.9",
    "matplotlib": "matplotlib>=3.8,<4",
    "openpyxl": "openpyxl>=3.1,<4",
}
live_requirements = {
    "requests": "requests>=2.32,<3",
    "tqdm": "tqdm>=4.66,<5",
    "Bio": "biopython>=1.84,<2",
    "rapidfuzz": "rapidfuzz>=3.9,<4",
    "bertopic": "bertopic>=0.17,<0.18",
    "sentence_transformers": "sentence-transformers>=5,<6",
    "umap": "umap-learn>=0.5,<0.6",
    "hdbscan": "hdbscan>=0.8,<0.9",
}
requirements = dict(snapshot_requirements)
if RUN_MODE.upper() == "LIVE_REFRESH":
    requirements.update(live_requirements)

missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")

In [ ]:
# Locate/download/upload the companion input ZIP, extract it, and load the
# versioned pipeline module stored inside the checksummed bundle.
from pathlib import Path
import importlib.util
import os
import shutil
import sys
import urllib.request
import zipfile

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

candidates = []
override = os.environ.get("MUSHROOM_UPSTREAM_INPUT_ZIP", "").strip()
if override:
    candidates.append(Path(override))
candidates.extend([Path.cwd() / INPUT_ZIP_NAME, Path("/content") / INPUT_ZIP_NAME])
input_zip = next((path for path in candidates if path.exists()), None)

if input_zip is None:
    raw_url = (
        "https://raw.githubusercontent.com/romenmeitei/"
        "AISKG_01_Framework/"
        "main/Mushroom_KG_Upstream_Inputs_v1.zip"
    )
    try:
        print("Trying repository companion bundle:", raw_url)
        urllib.request.urlretrieve(raw_url, INPUT_ZIP_NAME)
        input_zip = Path(INPUT_ZIP_NAME)
    except Exception as download_error:
        print("Automatic repository download was unavailable:", download_error)
        if IN_COLAB:
            print(f"Upload {INPUT_ZIP_NAME}")
            uploaded = colab_files.upload()
            if INPUT_ZIP_NAME in uploaded:
                Path(INPUT_ZIP_NAME).write_bytes(uploaded[INPUT_ZIP_NAME])
                input_zip = Path(INPUT_ZIP_NAME)
            elif uploaded:
                name, content = next(iter(uploaded.items()))
                input_zip = Path(name).name
                Path(input_zip).write_bytes(content)
        if input_zip is None:
            raise FileNotFoundError(
                f"Could not locate {INPUT_ZIP_NAME}. Place it beside the notebook or upload it in Colab."
            ) from download_error

WORK_ROOT = Path("/content/mushroom_kg_upstream_v1") if IN_COLAB else Path.cwd() / "mushroom_kg_upstream_run_v1"
INPUT_EXTRACT_ROOT = WORK_ROOT / "inputs"
OUTPUT_ROOT = WORK_ROOT / "outputs"
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
INPUT_EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(input_zip, "r") as archive:
    archive.extractall(INPUT_EXTRACT_ROOT)

manifests = sorted(INPUT_EXTRACT_ROOT.rglob("input_checksums.csv"), key=lambda p: (len(p.parts), str(p)))
root_manifests = [p for p in manifests if p.parent == INPUT_EXTRACT_ROOT]
if len(root_manifests) != 1:
    raise RuntimeError(f"Expected exactly one root input_checksums.csv; found {len(root_manifests)}")
INPUT_ROOT = root_manifests[0].parent
CORE_PATH = INPUT_ROOT / "code" / "upstream_core.py"
if not CORE_PATH.exists():
    raise FileNotFoundError(f"Missing pipeline module: {CORE_PATH}")

spec = importlib.util.spec_from_file_location("mushroom_upstream_core", CORE_PATH)
core = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = core
spec.loader.exec_module(core)

print("Input ZIP:", Path(input_zip).resolve())
print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Pipeline version:", core.PIPELINE_VERSION)

In [ ]:
# Execute the selected workflow. The default route is a complete offline replay.
import json
import os
from pathlib import Path


def read_secret(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

mode = RUN_MODE.upper().strip()
if mode == "MANUSCRIPT_SNAPSHOT":
    result = core.run_snapshot_pipeline(INPUT_ROOT, OUTPUT_ROOT)
    EFFECTIVE_MODE = mode
elif mode == "LIVE_REFRESH":
    config = core.LiveConfig()
    review_path = Path(LIVE_EXPERT_REVIEW_FILE).expanduser() if LIVE_EXPERT_REVIEW_FILE else None
    try:
        result = core.run_live_refresh(
            OUTPUT_ROOT,
            phase=LIVE_PHASE,
            config=config,
            entrez_email=read_secret("ENTREZ_EMAIL"),
            entrez_api_key=read_secret("NCBI_API_KEY"),
            scopus_api_key=read_secret("SCOPUS_API_KEY"),
            wos_api_key=read_secret("WOS_API_KEY"),
            annotated_topic_file=review_path,
            ontology_aliases=INPUT_ROOT / "data" / "ontology" / "ontology_entity_aliases.csv",
            relation_rules_file=INPUT_ROOT / "data" / "ontology" / "relation_rules.json",
        )
        live_archive = WORK_ROOT / OUTPUT_ZIP_NAME
        core.deterministic_zip(OUTPUT_ROOT, live_archive)
        result["output_archive"] = str(live_archive)
        (OUTPUT_ROOT / "LIVE_REFRESH_STAGE_SUCCESS.txt").write_text(
            f"LIVE_REFRESH {LIVE_PHASE} completed successfully.\n", encoding="utf-8"
        )
        EFFECTIVE_MODE = mode
    except Exception as live_error:
        if not FALLBACK_TO_SNAPSHOT:
            raise
        print("Live refresh did not complete:", repr(live_error))
        print("FALLBACK_TO_SNAPSHOT=True; running the frozen manuscript replay instead.")
        if OUTPUT_ROOT.exists():
            import shutil
            shutil.rmtree(OUTPUT_ROOT)
        result = core.run_snapshot_pipeline(INPUT_ROOT, OUTPUT_ROOT)
        result["live_refresh_error"] = repr(live_error)
        EFFECTIVE_MODE = "MANUSCRIPT_SNAPSHOT_FALLBACK"
else:
    raise ValueError("RUN_MODE must be MANUSCRIPT_SNAPSHOT or LIVE_REFRESH")

print("Effective mode:", EFFECTIVE_MODE)
print(json.dumps(result.get("run_manifest", result), indent=2, default=str))

In [ ]:
# Show the audit results and verify the bridge handed to Section 2.
from pathlib import Path
import pandas as pd
import zipfile

if EFFECTIVE_MODE.startswith("MANUSCRIPT_SNAPSHOT"):
    audit_files = [
        ("Input checksums", OUTPUT_ROOT / "audits" / "01_input_checksum_audit.csv"),
        ("Frozen output comparison", OUTPUT_ROOT / "audits" / "02_reference_output_audit.csv"),
        ("Fixed numerical checks", OUTPUT_ROOT / "audits" / "03_fixed_result_checks.csv"),
        ("Section 2 bridge checks", OUTPUT_ROOT / "audits" / "04_post_extraction_bridge_audit.csv"),
    ]
    for title, path in audit_files:
        frame = pd.read_csv(path)
        print(f"\n{title}: {len(frame)} rows")
        if "status" in frame.columns:
            print(frame["status"].value_counts(dropna=False).to_dict())
        try:
            display(frame)
        except NameError:
            print(frame.head())

    success_path = OUTPUT_ROOT / "PIPELINE_SUCCESS.txt"
    if not success_path.exists():
        raise RuntimeError("The snapshot run did not create PIPELINE_SUCCESS.txt")
    print("\n", success_path.read_text().strip())

    bridge_zip = OUTPUT_ROOT / BRIDGE_ZIP_NAME
    if not bridge_zip.exists():
        raise FileNotFoundError(f"Section 2 bridge ZIP was not created: {bridge_zip}")
    with zipfile.ZipFile(bridge_zip) as archive:
        bridge_names = sorted(name for name in archive.namelist() if not name.endswith("/"))
    print(f"Section 2 bridge: {bridge_zip.name} ({len(bridge_names)} files)")
    print("Core regenerated files:")
    for name in bridge_names:
        if any(name.endswith(core_name) for core_name in core.KEY_POST_EXTRACTION_FILES):
            print(" -", name)
else:
    print("Live-stage outputs:")
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            print(" -", path.relative_to(OUTPUT_ROOT))

In [ ]:
# Download the complete upstream output and, for snapshot runs, the ready-to-use
# Section 2 companion input ZIP.
from pathlib import Path

output_archive = Path(result["output_archive"])
files_to_offer = [output_archive]
if EFFECTIVE_MODE.startswith("MANUSCRIPT_SNAPSHOT"):
    files_to_offer.append(OUTPUT_ROOT / BRIDGE_ZIP_NAME)

if AUTO_DOWNLOAD_OUTPUTS:
    try:
        from google.colab import files
        for path in files_to_offer:
            files.download(str(path))
    except Exception:
        for path in files_to_offer:
            print("Available at:", path.resolve())
else:
    for path in files_to_offer:
        print("Available at:", path.resolve())

## Continue to Section 2

For the exact manuscript workflow, open the repository's second notebook and use the generated:

`Mushroom_KG_Reproducibility_Inputs_v2_from_upstream.zip`

This bridge contains the regenerated corpus/theme, entity, sentence-level relation, and aggregated-edge tables plus the frozen expert-validation and held-out benchmark inputs required by the already-tested canonical post-extraction pipeline.

### Optional live-refresh checkpoint

`LIVE_REFRESH / RETRIEVE_AND_MODEL` writes `16_live_topic_expert_review_template.csv`. A domain expert must complete its `expert_label`, `include_exclude`, and `comment` fields. In the same Colab session, set `LIVE_PHASE = "APPLY_CURATION_AND_EXTRACT"`, set `LIVE_EXPERT_REVIEW_FILE` to the completed CSV, and rerun the execution cell. Live outputs are dated updates and should not overwrite the frozen manuscript snapshot.

Section 2 repository: https://github.com/romenmeitei/AISKG_02_Framework
